# CNN in PyTorch

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [3]:
torch.manual_seed(42)

In [4]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [5]:
df = pd.read_csv('data/fashion-mnist_train.csv')
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [8]:
X_train = X_train/255.0
X_test = X_test/255.0

In [9]:
#defining the dataset
class CustomDataset(Dataset):
    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28)
        self.label = torch.tensor(label, dtype=torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self,index):
        return self.features[index], self.label[index]

In [10]:
train_dataset = CustomDataset(X_train, y_train)
#create test dataset
test_dataset = CustomDataset(X_test,y_test)

In [12]:
#create train and test loader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [17]:
class Mynn(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features, out_channels=32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, out_channels=64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)

        return x


In [18]:
#set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [19]:
#instantiate the model
model = Mynn(1)
model = model.to(device)
#loss function 
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [20]:
#training loop

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_label in train_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        #forwardpass
        outputs = model(batch_features)
        #calculate loss
        loss = criterion(outputs, batch_label)
        #backpass
        optimizer.zero_grad()
        loss.backward()
        #updateparams
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch+1}, Loss:{avg_loss:.4f}')
    

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch: 1, Loss:0.5873
Epoch: 2, Loss:0.4014
Epoch: 3, Loss:0.3436
Epoch: 4, Loss:0.3064
Epoch: 5, Loss:0.2823
Epoch: 6, Loss:0.2624
Epoch: 7, Loss:0.2424
Epoch: 8, Loss:0.2288
Epoch: 9, Loss:0.2178
Epoch: 10, Loss:0.2045
Epoch: 11, Loss:0.1999
Epoch: 12, Loss:0.1879
Epoch: 13, Loss:0.1825
Epoch: 14, Loss:0.1747
Epoch: 15, Loss:0.1723
Epoch: 16, Loss:0.1638
Epoch: 17, Loss:0.1589
Epoch: 18, Loss:0.1500
Epoch: 19, Loss:0.1450
Epoch: 20, Loss:0.1433
Epoch: 21, Loss:0.1389
Epoch: 22, Loss:0.1339
Epoch: 23, Loss:0.1336
Epoch: 24, Loss:0.1304
Epoch: 25, Loss:0.1235
Epoch: 26, Loss:0.1259
Epoch: 27, Loss:0.1196
Epoch: 28, Loss:0.1200
Epoch: 29, Loss:0.1158
Epoch: 30, Loss:0.1107
Epoch: 31, Loss:0.1116
Epoch: 32, Loss:0.1125
Epoch: 33, Loss:0.1079
Epoch: 34, Loss:0.0988
Epoch: 35, Loss:0.0993
Epoch: 36, Loss:0.0976
Epoch: 37, Loss:0.0948
Epoch: 38, Loss:0.0965
Epoch: 39, Loss:0.0951
Epoch: 40, Loss:0.0930
Epoch: 41, Loss:0.0883
Epoch: 42, Loss:0.0967
Epoch: 43, Loss:0.0849
Epoch: 44, Loss:0.08

In [21]:
#set model to eval mode

model.eval()

Mynn(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [22]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in test_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        output = model(batch_features)
        _, predicted = torch.max(output,1)
        
        total = total + batch_label.shape[0]
        correct = correct + (predicted == batch_label).sum().item()
    
print(correct/total)


0.9213333333333333


In [23]:
#evaluation code

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in train_loader:
        batch_features, batch_label = batch_features.to(device), batch_label.to(device)
        output = model(batch_features)
        _, predicted = torch.max(output,1)
        
        total = total + batch_label.shape[0]
        correct = correct + (predicted == batch_label).sum().item()
    
print(correct/total)


0.9964583333333333
